In [ ]:
# ==========================================
# CELL 1: Standard Imports & Paths
# ==========================================
import os
import sys

sys.path.append('scripts')

import torch
import types
from esp_ppq.api import get_target_platform
import esp_ppq.lib as PFL
from esp_ppq.executor import TorchExecutor
from esp_ppq.core import QuantizationVisibility, TargetPlatform
from esp_ppq.api.interface import load_onnx_graph
from esp_ppq.quantization.optim import (
    QuantizeSimplifyPass, QuantizeFusionPass, ParameterQuantizePass,
    RuntimeCalibrationPass, PassiveParameterQuantizePass, QuantAlignmentPass,
    TrainedQuantizationThresholdPass
)

In [ ]:
# CONFIGURACION HAMMER + PLIER PARA ESP32-S3

IMG_SZ_I = 256
PLATFORM = "s3"
PROJECT_NAME = "hammer_plier"
DATA_YAML_FILE_I = r"C:/YOLO/hammer_plier_calib.yaml"
INT16_LUT_STEP_I = 32

print('CONFIGURACION:')
print(' Plataforma:', PLATFORM)
print(' Resolucion:', IMG_SZ_I)
print(' Dataset:', DATA_YAML_FILE_I)


In [ ]:
class QATConfig:
    IMG_SZ = IMG_SZ_I
    DEVICE = "cuda" if torch.cuda.is_available() and torch.cuda.device_count() > 0 else "cpu"
    DATA_YAML_FILE = DATA_YAML_FILE_I

    BATCH_SIZE = 1
    CALIB_MAX_IMAGES = 64
    CALIB_VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    DATA_FALLBACK_PATH = r"C:/YOLO/calib_hammer_plier"

    CALIB_STEPS = 32
    QUANT_CALIB_METHOD = "percentile"
    QUANT_ALIGNMENT = "Align to Output"
    TARGET_PLATFORM = get_target_platform("esp32" + PLATFORM, 8)

    INT16_LUT_STEP = INT16_LUT_STEP_I

    TQT_STEPS = 200
    TQT_LR = 1e-5
    TQT_INT_LAMBDA = 0.25
    TQT_BLOCK_SIZE = 4
    TQT_COLLECTING_DEVICE = "cpu"

    BASE_DIR = os.getcwd()
    MODEL_NAME = "yolo26n_hammer_plier_core_reforzado"
    PT_FILE = f"{MODEL_NAME}.pt"
    ONNX_FILE = f"{MODEL_NAME}_export.onnx"
    ESPDL_OUTPUT_DIR = os.path.join(
        BASE_DIR, "output", f"hammer_plier_core_reforzado_{IMG_SZ_I}_s8_{PLATFORM}"
    )
    ONNX_PATH = os.path.join(ESPDL_OUTPUT_DIR, ONNX_FILE)

if 'config' not in sys.modules:
    sys.modules['config'] = types.ModuleType('config')
sys.modules['config'].QATConfig = QATConfig

print('DEVICE:', QATConfig.DEVICE)
print('BATCH_SIZE:', QATConfig.BATCH_SIZE)
print('MODEL:', QATConfig.PT_FILE)


In [ ]:
# ==========================================
# CELL 4: Local Modules & Environment Setup
# ==========================================
from utils import seed_everything, register_mod_op, get_exclusive_ancestors
from dataset import get_calibration_loader
from ultralytics.data.utils import check_det_dataset
from esp_ppq_patch import apply_esp_ppq_patches
from esp_ppq_patch_2 import apply_addlut_patch
from notebook_helpers import extract_model_meta, prepare_onnx, prune_graph_safely, eval_espdl_model
from esp_ppq_lut.passes import EspdlLUTFusionPass
from esp_ppq_lut.exporter import HardwareAwareEspdlExporter

os.makedirs(QATConfig.ESPDL_OUTPUT_DIR, exist_ok=True)

import esp_ppq_lut as esp_lut
esp_lut.initialize(step=QATConfig.INT16_LUT_STEP, verbose=True)

seed_everything(1234)
register_mod_op()
apply_esp_ppq_patches()
apply_addlut_patch()

print("Environment and Configuration setup complete.")

In [ ]:
# ==========================================
# CELL 5: ONNX Export & Metadata Extraction
# ==========================================
prepare_onnx()
model_meta = extract_model_meta()

In [ ]:
# ==========================================
# CELL 6: Quantizer Initialization & Branch Separation
# ==========================================
print("Loading ONNX Graph into ESP-PPQ...")
graph = load_onnx_graph(onnx_import_file=QATConfig.ONNX_PATH)

output_names = list(graph.outputs.keys())
aux_ops = set()
main_ops = set()

if len(output_names) >= 6:
    aux_outputs = output_names[0:3]
    main_outputs = output_names[3:6]
    aux_ops = get_exclusive_ancestors(graph, aux_outputs, main_outputs)
    main_ops = get_exclusive_ancestors(graph, main_outputs, aux_outputs)

quantizer = PFL.Quantizer(platform=QATConfig.TARGET_PLATFORM, graph=graph)
dispatching_table = PFL.Dispatcher(graph=graph, method="conservative").dispatch(
    quantizer.quant_operation_types
)

for opname, platform in dispatching_table.items():
    if platform == TargetPlatform.UNSPECIFIED:
        dispatching_table[opname] = TargetPlatform(quantizer.target_platform)

for op in aux_ops:
    if op.name in dispatching_table:
        dispatching_table[op.name] = TargetPlatform.FP32

In [ ]:
# ==========================================
# CELL 7: Hardware Precision Targets (INT16 / FP32)
# ==========================================
INT16_PLATFORM = get_target_platform("esp32" + PLATFORM, 16)

# Force high-sensitivity exit layers to INT16
int16_layers = {
    # Neck Exits
    "/model.16/cv2/conv/Conv", "/model.16/cv2/conv/Conv/Swish",
    "/model.19/cv2/conv/Conv", "/model.19/cv2/conv/Conv/Swish",
    "/model.22/cv2/conv/Conv", "/model.22/cv2/conv/Conv/Swish",

    # Box Heads
    "/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv", "/model.23/one2one_cv2.0/one2one_cv2.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv", "/model.23/one2one_cv2.0/one2one_cv2.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.0/one2one_cv2.0.2/Conv",
    "/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv", "/model.23/one2one_cv2.1/one2one_cv2.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv", "/model.23/one2one_cv2.1/one2one_cv2.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.1/one2one_cv2.1.2/Conv",
    "/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv", "/model.23/one2one_cv2.2/one2one_cv2.2.0/conv/Conv/Swish",
    "/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv", "/model.23/one2one_cv2.2/one2one_cv2.2.1/conv/Conv/Swish",
    "/model.23/one2one_cv2.2/one2one_cv2.2.2/Conv",

    # Class Heads
    "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.0/one2one_cv3.0.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv", "/model.23/one2one_cv3.0/one2one_cv3.0.1/one2one_cv3.0.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.0/one2one_cv3.0.2/Conv",
    "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.0/one2one_cv3.1.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv", "/model.23/one2one_cv3.1/one2one_cv3.1.1/one2one_cv3.1.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.1/one2one_cv3.1.2/Conv",
    "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.0/one2one_cv3.2.0.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.0/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv", "/model.23/one2one_cv3.2/one2one_cv3.2.1/one2one_cv3.2.1.1/conv/Conv/Swish",
    "/model.23/one2one_cv3.2/one2one_cv3.2.2/Conv"
}

for op in graph.operations.values():
    if op.name in dispatching_table and op.name in int16_layers:
        dispatching_table[op.name] = INT16_PLATFORM

# Workaround FP32 Concat Limits
fp32_layers = {"/model.23/Concat_5", "/model.23/Concat_3", "/model.23/Concat_4"}
for op in main_ops:
    if op.name in fp32_layers:
        dispatching_table[op.name] = TargetPlatform.FP32

print("Applying Dispatcher Types...")
for op in graph.operations.values():
    quantizer.quantize_operation(op_name=op.name, platform=dispatching_table[op.name])

In [ ]:
# ==========================================
# CELL 8: Linear Optimization Pipeline (PTQ + TQT + LUT)
# BATCH 1 PARA ESP32-S3
# ==========================================
print("Running Linear Optimization Pipeline (Calibration -> TQT -> LUT)...")
data_cfg = check_det_dataset(QATConfig.DATA_YAML_FILE)
cali_loader = get_calibration_loader(data_cfg)

executor = TorchExecutor(graph=graph, device=QATConfig.DEVICE)

dummy_input = torch.zeros(
    [1, 3, QATConfig.IMG_SZ, QATConfig.IMG_SZ]
).to(QATConfig.DEVICE)
executor.tracing_operation_meta(inputs=dummy_input)

pipeline = PFL.Pipeline([
    QuantizeSimplifyPass(),
    QuantizeFusionPass(activation_type=quantizer.activation_fusion_types),
    ParameterQuantizePass(),
    RuntimeCalibrationPass(method=QATConfig.QUANT_CALIB_METHOD),

    TrainedQuantizationThresholdPass(
        steps=QATConfig.TQT_STEPS,
        lr=QATConfig.TQT_LR,
        int_lambda=QATConfig.TQT_INT_LAMBDA,
        block_size=QATConfig.TQT_BLOCK_SIZE,
        collecting_device=QATConfig.TQT_COLLECTING_DEVICE
    ),

    PassiveParameterQuantizePass(
        clip_visiblity=QuantizationVisibility.EXPORT_WHEN_ACTIVE
    ),
    QuantAlignmentPass(
        elementwise_alignment=QATConfig.QUANT_ALIGNMENT
    ),

    EspdlLUTFusionPass(
        target_ops=['Swish'],
        lut_step=QATConfig.INT16_LUT_STEP
    )
])

pipeline.optimize(
    calib_steps=QATConfig.CALIB_STEPS,
    collate_fn=(lambda x: x.type(torch.float).to(QATConfig.DEVICE)),
    graph=graph,
    dataloader=cali_loader,
    executor=executor,
)
print("Pipeline complete.")


In [ ]:
print('Validacion mAP omitida para este proceso.')


In [ ]:
# ==========================================
# CELL 10: Graph Surgery (Pruning and Output Tearing)
# ==========================================
print("Slicing output Concat nodes into 6 discrete tensors...")

# 1. Remove Aux Heads
output_names = list(graph.outputs.keys())
if len(output_names) >= 6:
    for name in output_names[0:3]:
        if name in graph.outputs: graph.outputs.pop(name)
    prune_graph_safely(graph)

# 2. Slice the Concat into Box/Cls
targets = ["one2one_p3", "one2one_p4", "one2one_p5"]
collected_outputs = {}
for target_name in targets:
    if target_name in graph.outputs:
        original_output_var = graph.variables[target_name]
        producer = original_output_var.source_op

        if producer and producer.type == "Concat":
            box_var, cls_var = None, None
            for input_var in producer.inputs:
                dims = input_var.shape
                if dims is not None:
                    if 4 in dims: box_var = input_var
                    elif model_meta['nc'] in dims: cls_var = input_var

            if box_var and cls_var:
                pair_config = [
                    (box_var, f"{target_name}_box"),
                    (cls_var, f"{target_name}_cls"),
                ]
                for var, new_name in pair_config:
                    old_name = var.name
                    if old_name in graph.variables: graph.variables.pop(old_name)
                    var._name = new_name
                    graph.variables[new_name] = var
                    collected_outputs[new_name] = var

                graph.outputs.pop(target_name)
                graph.remove_operation(producer, keep_coherence=False)
                for var in producer.inputs:
                    if producer in var.dest_ops: var.dest_ops.remove(producer)

# 3. Enforce precise output order matching ESPdl C++ expectations
final_output_list = [
    "one2one_p3_box", "one2one_p3_cls",
    "one2one_p4_box", "one2one_p4_cls",
    "one2one_p5_box", "one2one_p5_cls"
]
graph.outputs.clear()
for name in final_output_list:
    if name in collected_outputs:
        graph.outputs[name] = collected_outputs[name]

prune_graph_safely(graph)


In [ ]:
print('Preview omitida.')


In [ ]:
# ==========================================
# CELL 11: Final Export
# ==========================================
final_espdl_path = os.path.join(QATConfig.ESPDL_OUTPUT_DIR, f"{QATConfig.MODEL_NAME}_{QATConfig.IMG_SZ}_s8_{PLATFORM}.espdl")

# Using the properly initialized local exporter
exporter = PFL.Exporter(platform=QATConfig.TARGET_PLATFORM)
exporter.export(final_espdl_path, graph=graph, int16_lut_step=QATConfig.INT16_LUT_STEP)
print(f"Deployment Model exported seamlessly to {final_espdl_path}")

In [ ]:
print("PRUEBA GRAFO CUANTIZADO - PLAYO REAL CORES3")

predicciones, ruta_salida = eval_espdl_model(
    test_image_path=r"C:\\YOLO\\debug_entrada_yolo_playo.jpg",
    graph=graph,
    target_img_sz=QATConfig.IMG_SZ,
    data_yaml=QATConfig.DATA_YAML_FILE,
    platform=PLATFORM,
    conf_thresh=0.001,
    iou_thresh=0.45,
    output_dir="results_core_reforzado_playo_real",
)

print("\nPREDICCIONES PLAYO REAL:")
print(predicciones)
print("\nSALIDA:")
print(ruta_salida)
